In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
import os
import random
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ====================== 随机种子 ======================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

LOCAL_MODEL_DIR = "./hf_models"


# ====================== 数据加载 ======================
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

# ====================== 共享 Dataset ======================
class AblationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }
        return item

# ====================== 模型 1: Baseline ======================
class BaselineModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, 4)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}




# ====================== 共享训练函数 ======================
def train_and_evaluate(model_class, model_name, use_class_weight=False, use_multilabel=False, use_hierarchical=False, use_ordinal=False):
    print(f"\n{'='*50}")
    print(f"开始训练模型: {model_name}")
    print(f"{'='*50}")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    
    train_dataset = AblationDataset(train_texts, train_labels, tokenizer)
    val_dataset   = AblationDataset(val_texts,   val_labels,   tokenizer)
    test_dataset  = AblationDataset(test_texts,  test_labels,  tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)
    
    model = model_class(MODEL_PATH).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    
    # 计算 class weights
    if use_class_weight:
        from sklearn.utils.class_weight import compute_class_weight
        class_weights_np = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
        class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(DEVICE)
    else:
        class_weights = None
    
    best_f1 = 0
    best_state = None
    
    for epoch in range(5):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            if use_multilabel:
                # 多标签额外处理标签
                multi_labels = torch.zeros((labels.size(0), 3), device=DEVICE)
                sev_labels = torch.zeros(labels.size(0), device=DEVICE)
                for i, l in enumerate(labels):
                    if l == 0: multi_labels[i,0] = 1; sev_labels[i] = 0.25
                    elif l == 1: multi_labels[i,1] = 1; sev_labels[i] = 0.55
                    elif l == 3: multi_labels[i,1] = 1; multi_labels[i,2] = 1; sev_labels[i] = 0.92
                outputs = model(input_ids, attention_mask, multi_labels=multi_labels, severity_labels=sev_labels)
            elif use_hierarchical or use_ordinal:
                outputs = model(input_ids, attention_mask, labels=labels)
            elif use_class_weight:
                outputs = model(input_ids, attention_mask, labels=labels, class_weights=class_weights)
            else:
                # Baseline 普通模型
                outputs = model(input_ids, attention_mask, labels=labels)
            
            loss = outputs["loss"]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1} Avg Loss: {total_loss / len(train_loader):.4f}")
        
        # Validation
        model.eval()
        all_preds = []
        all_labels_list = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                
                if use_multilabel:
                    outputs = model(input_ids, attention_mask)
                    probs = torch.sigmoid(outputs["symptom_logits"])
                    risks = outputs["risk_score"]
                    preds = []
                    for p, r in zip(probs, risks):
                        if r > 0.65: preds.append(3)
                        elif p[1] > 0.60: preds.append(1)
                        elif p[0] > 0.55: preds.append(0)
                        else: preds.append(2)
                    preds = torch.tensor(preds, device=DEVICE)
                elif use_hierarchical:
                    preds = model.predict(input_ids, attention_mask)
                elif use_ordinal:
                    preds = model.predict(input_ids, attention_mask)
                else:
                    outputs = model(input_ids, attention_mask, labels=None)
                    preds = torch.argmax(outputs["logits"], dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels_list.extend(labels.cpu().numpy())
        
        f1 = f1_score(all_labels_list, all_preds, average="weighted")
        print(f"Epoch {epoch+1} Val Weighted F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
    
    # 保存最佳模型
    torch.save(best_state, f"best_{model_name.replace(' ', '_')}.pt")
    print(f"Best model saved: best_{model_name.replace(' ', '_')}.pt (Val F1: {best_f1:.4f})")
    
    # Test
    model.load_state_dict(best_state)
    model.eval()
    test_preds = []
    test_labels_list = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            if use_multilabel:
                outputs = model(input_ids, attention_mask)
                probs = torch.sigmoid(outputs["symptom_logits"])
                risks = outputs["risk_score"]
                preds = []
                for p, r in zip(probs, risks):
                    if r > 0.65: preds.append(3)
                    elif p[1] > 0.60: preds.append(1)
                    elif p[0] > 0.55: preds.append(0)
                    else: preds.append(2)
                preds = torch.tensor(preds, device=DEVICE)
            elif use_hierarchical:
                preds = model.predict(input_ids, attention_mask)
            elif use_ordinal:
                preds = model.predict(input_ids, attention_mask)
            else:
                outputs = model(input_ids, attention_mask, labels=None)
                preds = torch.argmax(outputs["logits"], dim=1)
            
            test_preds.extend(preds.cpu().numpy())
            test_labels_list.extend(labels.cpu().numpy())
    
    print(f"\n=== Test Results: {model_name} ===")
    report = classification_report(test_labels_list, test_preds, target_names=label_encoder.classes_, digits=4)
    print(report)
    
    cm = confusion_matrix(test_labels_list, test_preds)
    print("Confusion Matrix:")
    print(cm)
    
    # 保存报告
    with open(f"{model_name.replace(' ', '_')}_test_report.txt", "w", encoding="utf-8") as f:
        f.write(report)
        f.write("\nConfusion Matrix:\n")
        f.write(str(cm))
    
    return {
        "model": model_name,
        "accuracy": accuracy_score(test_labels_list, test_preds),
        "weighted_f1": f1_score(test_labels_list, test_preds, average="weighted"),
        "macro_f1": f1_score(test_labels_list, test_preds, average="macro"),
        "dep_f1": classification_report(test_labels_list, test_preds, target_names=label_encoder.classes_, output_dict=True)['Depression']['f1-score'],
        "sui_f1": classification_report(test_labels_list, test_preds, target_names=label_encoder.classes_, output_dict=True)['Suicidal']['f1-score'],
        "dep_to_sui": cm[1][3] if len(cm) > 1 else 0,  # Depression → Suicidal
        "sui_to_dep": cm[3][1] if len(cm) > 3 else 0   # Suicidal → Depression
    }

# ====================== 依次运行 4 个模型 ======================
results = []

# 1. roberta-base
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")
res = train_and_evaluate(BaselineModel, "roberta-base")
results.append(res)

# 2. mental-bert-base-uncased
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "mental-bert-base-uncased")
res = train_and_evaluate(BaselineModel, "mental-bert-base-uncased")
results.append(res)

# 3. bert-base-uncased
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "bert-base-uncased")
res = train_and_evaluate(BaselineModel, "bert-base-uncased")
results.append(res)

# 4. distilbert-base-uncased
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "distilbert-base-uncased")
res = train_and_evaluate(BaselineModel, "distilbert-base-uncased")
results.append(res)


# ====================== 生成对比表 ======================
print("\n" + "="*60)
print("Baseline Model Comparison Table")
print("="*60)
print(f"{'Model':<30} {'Acc':<8} {'W-F1':<8} {'M-F1':<8} {'Dep F1':<8} {'Sui F1':<8} {'Dep→Sui':<10} {'Sui→Dep':<10}")
print("-"*80)
for r in results:
    print(f"{r['model']:<30} {r['accuracy']:<8.4f} {r['weighted_f1']:<8.4f} {r['macro_f1']:<8.4f} {r['dep_f1']:<8.4f} {r['sui_f1']:<8.4f} {r['dep_to_sui']:<10} {r['sui_to_dep']:<10}")

# 保存到文件
with open("baseline_comparison.txt", "w", encoding="utf-8") as f:
    f.write("Baseline Model Comparison Table\n")
    f.write("="*60 + "\n")
    f.write(f"{'Model':<30} {'Acc':<8} {'W-F1':<8} {'M-F1':<8} {'Dep F1':<8} {'Sui F1':<8} {'Dep→Sui':<10} {'Sui→Dep':<10}\n")
    f.write("-"*80 + "\n")
    for r in results:
        f.write(f"{r['model']:<30} {r['accuracy']:<8.4f} {r['weighted_f1']:<8.4f} {r['macro_f1']:<8.4f} {r['dep_f1']:<8.4f} {r['sui_f1']:<8.4f} {r['dep_to_sui']:<10} {r['sui_to_dep']:<10}\n")

print("\n训练完成！对比表已保存为 baseline_comparison.txt")
print("每个模型的最佳 checkpoint 已保存为 best_*.pt")
print("每个模型的测试报告已保存为 *_test_report.txt")


Using device: cuda

开始训练模型: roberta-base


Loading weights: 100%|██| 197/197 [00:00<00:00, 580.22it/s, Materializing param=encoder.layer.11.output.dense.weight]
RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1: 100%|███████████████████████████████████████████████████████████████████| 2171/2171 [13:0

Epoch 1 Avg Loss: 0.4525
Epoch 1 Val Weighted F1: 0.8640


Epoch 2: 100%|███████████████████████████████████████████████████████████████████| 2171/2171 [15:29<00:00,  2.34it/s]


Epoch 2 Avg Loss: 0.3145
Epoch 2 Val Weighted F1: 0.8619


...

Epoch 3 Avg Loss: 0.2558
Epoch 3 Val Weighted F1: 0.8325


Epoch 4: 100%|███████████████████████████████████████████████████████████████████| 2171/2171 [07:45<00:00,  4.66it/s]


Epoch 4 Avg Loss: 0.1748
Epoch 4 Val Weighted F1: 0.8143


Epoch 5: 100%|███████████████████████████████████████████████████████████████████| 2171/2171 [07:51<00:00,  4.60it/s]


Epoch 5 Avg Loss: 0.1136
Epoch 5 Val Weighted F1: 0.8226
Best model saved: best_distilbert-base-uncased.pt (Val F1: 0.8325)

=== Test Results: distilbert-base-uncased ===
              precision    recall  f1-score   support

     Anxiety     0.8632    0.8172    0.8396       826
  Depression     0.7355    0.7169    0.7261      2176
      Normal     0.9474    0.9271    0.9371      2758
    Suicidal     0.6967    0.7622    0.7280      1682

    accuracy                         0.8162      7442
   macro avg     0.8107    0.8059    0.8077      7442
weighted avg     0.8194    0.8162    0.8173      7442

Confusion Matrix:
[[ 675  101   38   12]
 [  49 1560   60  507]
 [  52  110 2557   39]
 [   6  350   44 1282]]

Baseline Model Comparison Table
Model                          Acc      W-F1     M-F1     Dep F1   Sui F1   Dep→Sui    Sui→Dep   
--------------------------------------------------------------------------------
roberta-base                   0.8581   0.8584   0.8497   0.7844   0.77